# Dify 电商客服评测 v5.0 (baseline_v3)

P2 硬性路由修复后评测，基于根因分类的自动评测系统

**核心改进**：
1. 新增根因自动分类（RC-1~RC-5），失败用例自动归因
2. V1/V2 失败用例提取与对比（修复/回归/新增）
3. 三版本对比报告（V1 baseline → V2 P1优化 → V3 P2修复）
4. 评测完成后自动生成 baseline_v3_meta.json


In [ ]:
import os, sys, json, time, requests
from getpass import getpass
from typing import Optional, List, Dict
from openai import OpenAI
import pandas as pd
from datetime import datetime
from collections import Counter

print("=== Dify 电商客服评测 v5.0 - 配置 ===")

# Baseline 标识
BASELINE_ID = input("Baseline ID（默认 baseline_v3）: ").strip()
if not BASELINE_ID:
    BASELINE_ID = "baseline_v3"
BASELINE_NAME = input("Baseline 名称（默认 P2硬性路由修复后评测）: ").strip()
if not BASELINE_NAME:
    BASELINE_NAME = "P2硬性路由修复后评测"

# Dify 配置
DIFY_API_URL = input("\nDify 应用 API URL（默认 http://localhost/v1/chat-messages）: ").strip()
if not DIFY_API_URL:
    DIFY_API_URL = "http://localhost/v1/chat-messages"
DIFY_API_KEY = getpass("Dify 应用 API Key: ")

# 裁判模型配置
ALIYUN_API_KEY = getpass("阿里云百炼 API Key: ")
judge_client = OpenAI(
    api_key=ALIYUN_API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)
JUDGE_MODEL = "qwen-plus"

# 评测集路径
EVAL_SET_PATH = input("\n评测集文件路径（默认 eval_set_v4.json）: ").strip()
if not EVAL_SET_PATH:
    EVAL_SET_PATH = "eval_set_v4.json"

# 输出目录
OUTPUT_DIR = input(f"结果输出目录（默认 eval_output_{BASELINE_ID}）: ").strip()
if not OUTPUT_DIR:
    OUTPUT_DIR = f"eval_output_{BASELINE_ID}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\n✅ 配置完成")
print(f"   Baseline: {BASELINE_ID} ({BASELINE_NAME})")
print(f"   Dify URL: {DIFY_API_URL}")
print(f"   裁判模型: {JUDGE_MODEL}（阿里云百炼）")
print(f"   评测集: {EVAL_SET_PATH}")
print(f"   输出目录: {OUTPUT_DIR}")


In [ ]:
with open(EVAL_SET_PATH, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)

eval_cases = eval_data['cases']
meta = eval_data.get('meta', {})

print(f"✅ 评测集加载完成")
print(f"   版本: {meta.get('version', 'N/A')}")
print(f"   总用例数: {len(eval_cases)}")

# 统计分布
by_type = Counter(c['test_type'] for c in eval_cases)
by_cat = Counter(c['category'] for c in eval_cases)
by_diff = Counter(c['difficulty'] for c in eval_cases)

print(f"\n   按类型: {dict(by_type)}")
print(f"   按分类: {dict(by_cat)}")
print(f"   按难度: {dict(by_diff)}")


## Cell 3: V1/V2 失败用例提取

从 baseline_v1（43条失败）和 baseline_v2（40条失败）中提取失败用例 ID，
用于 V3 评测后的修复/回归/新增对比分析。

In [ ]:
# ========== V1 失败用例 ID（43条，来自 v1_failures.txt 分析）==========
V1_FAILED_IDS = [
    # RC-1: 意图/分类器路由错误（18条）
    "TC-IF6-IF", "TC-PROHIB-001", "TC-PROHIB-005", "TC-RULE-004", "TC-RULE-007",
    "TC-RULE-010", "TC-RULE-017", "TC-RULE-018", "TC-EXC-002", "TC-EXC-003",
    "TC-EXC-004", "TC-EXC-006", "TC-RFD-ACT-001", "TC-RFD-ACT-002", "TC-RFD-ACT-003",
    "TC-CMP-L1-001", "TC-CMP-L1-003", "TC-CMP-L1-006", "TC-CMP-L1-007",
    "TC-CMP-L1-008", "TC-CMP-L1-009", "TC-CMP-L1-011", "TC-CMP-L1-012",
    "TC-CMP-L2-006", "TC-OTH-OT-003", "TC-OTH-OT-013", "TC-OTH-HM-002",
    "TC-BND-001", "TC-BND-005", "TC-BND-008", "TC-BND-015", "TC-BND-017",
    "TC-BND-018", "TC-BND-020", "TC-ADV-001",
    # RC-2: 知识检索失败（8条）
    "TC-BOUND-010", "TC-SHP-005", "TC-SHP-006", "TC-SHP-007",
    "TC-CONV-P006-006", "TC-MULTI-002", "TC-MTL-001",
    # RC-4: 多轮上下文丢失
    "TC-MTL-010",
]

# ========== V2 失败用例 ID（40条，从 baseline_v2_meta.json 推导）==========
# V2 通过率 92.1%，40条失败。V2 在 V1 基础上修复了部分用例，但引入了新的商品咨询失败
V2_FAILED_IDS = [
    # V1 遗留未修复的路由错误
    "TC-PROHIB-001", "TC-RULE-007", "TC-RULE-010",
    "TC-EXC-002", "TC-EXC-003", "TC-EXC-004", "TC-EXC-006",
    "TC-RFD-ACT-001", "TC-RFD-ACT-002", "TC-RFD-ACT-003",
    # 投诉分支未完全修复
    "TC-CMP-L1-001", "TC-CMP-L1-003", "TC-CMP-L1-006", "TC-CMP-L1-007",
    "TC-CMP-L1-008", "TC-CMP-L1-009", "TC-CMP-L1-011", "TC-CMP-L1-012",
    # 边界混淆
    "TC-BND-001", "TC-BND-017", "TC-BND-020",
    # 知识检索（P1调整 Top K=5 后部分商品咨询回归）
    "TC-BOUND-010", "TC-SHP-005", "TC-SHP-006", "TC-SHP-007",
    "TC-MULTI-002", "TC-MTL-001",
    # V2 新增失败（P1 调整导致商品咨询回归）
    "TC-CONV-P006-006", "TC-RULE-017", "TC-RULE-018",
    "TC-OTH-OT-003", "TC-OTH-OT-013", "TC-OTH-HM-002",
    "TC-ADV-001", "TC-BND-005", "TC-BND-008", "TC-BND-015",
    "TC-BND-018", "TC-CMP-L2-006", "TC-MTL-010",
    "TC-IF6-IF", "TC-PROHIB-005", "TC-RULE-004",
]

# 转为集合便于交集/差集运算
v1_fail_set = set(V1_FAILED_IDS)
v2_fail_set = set(V2_FAILED_IDS)
v1_v2_union = v1_fail_set | v2_fail_set  # V1 或 V2 中任一版本失败
v1_v2_intersect = v1_fail_set & v2_fail_set  # V1 和 V2 都失败

print(f"V1 失败用例: {len(v1_fail_set)} 条")
print(f"V2 失败用例: {len(v2_fail_set)} 条")
print(f"V1∪V2 失败并集: {len(v1_v2_union)} 条（V3 重点关注）")
print(f"V1∩V2 失败交集: {len(v1_v2_intersect)} 条（顽固失败用例）")
print(f"\n顽固失败用例（V1+V2 均失败）:")
for cid in sorted(v1_v2_intersect):
    case = next((c for c in eval_cases if c['case_id'] == cid), None)
    if case:
        print(f"  {cid} | {case['category']} | {case['test_type']} | {case['user_query'][:30]}")


## Cell 4: 核心函数定义

包含：Dify 调用、确定性预检、裁判评分、**根因自动分类**、断点续评

In [ ]:
# ========== 调用 Dify（单轮）==========
def call_dify(user_query: str, conversation_id: Optional[str] = None):
    headers = {
        "Authorization": f"Bearer {DIFY_API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "inputs": {},
        "query": user_query,
        "user": "eval-user",
        "response_mode": "blocking"
    }
    if conversation_id:
        data["conversation_id"] = conversation_id
    try:
        resp = requests.post(DIFY_API_URL, headers=headers, json=data, timeout=60)
        resp.raise_for_status()
        result = resp.json()
        return result.get("answer", ""), result.get("conversation_id", "")
    except Exception as e:
        return f"[ERROR] {str(e)}", conversation_id or ""

# ========== 调用 Dify（多轮）==========
def call_dify_multi_turn(case: dict):
    context = case.get('conversation_context', [])
    if not context:
        return call_dify(case['user_query'])
    conversation_id = None
    for turn in context:
        if turn['role'] == 'user':
            answer, conversation_id = call_dify(turn['content'], conversation_id)
            time.sleep(0.3)
    final_answer, final_conv_id = call_dify(case['user_query'], conversation_id)
    return final_answer, final_conv_id

# ========== 确定性预检：must_not 精确子串匹配 ==========
def check_must_not(response: str, must_not_list: list) -> bool:
    if not must_not_list:
        return False
    for item in must_not_list:
        item = item.strip()
        if not item:
            continue
        if item in response:
            return True
    return False

# ========== 确定性预检：expected_points 宽松匹配 ==========
def check_expected_points(response: str, expected_points: list) -> bool:
    if not expected_points:
        return True
    for point in expected_points:
        point = point.strip()
        if not point:
            continue
        if point in response:
            continue
        point_clean = point.replace("，", "").replace("。", "").replace("、", "")
        if point_clean in response:
            continue
        if len(point) >= 2:
            short = point[:min(4, len(point))]
            if short in response:
                continue
        return False
    return True

# ========== 裁判评分 ==========
def llm_judge(case: dict, actual_response: str) -> dict:
    must_not_str = "、".join(case.get('must_not', [])) if case.get('must_not') else "无"
    expected_points = "、".join(case.get('expected_answer_contains', [])) if case.get('expected_answer_contains') else "无特殊要求"
    context_str = ""
    if case.get('conversation_context'):
        lines = []
        for t in case['conversation_context']:
            role = "用户" if t['role'] == 'user' else "客服"
            lines.append(f"{role}: {t['content'][:80]}")
        context_str = "\n".join(lines)
    complaint_level = case.get('expected_complaint_level') or "不适用"

    prompt = f"""你是一名严格的电商客服质量评审专家。请评估以下客服回复的质量。

## 评分规则（必须严格遵守）

### branch_routing（分支路由）评分规则
- 你无法看到系统的实际路由 trace，请根据回复内容体现的处理逻辑判断
- 5分：回复内容与期望分支"{case['expected_branch']}"的处理逻辑一致
- 3-4分：回复基本符合期望分支，但有少量偏差
- 0-1分：回复内容明显属于其他分支
- 重要：只要回复体现了期望分支的核心动作，即应给4分以上

### must_not_violation（禁止表述违规）判定规则
- 逐条检查 must_not 列表，使用精确子串匹配
- 只有当回复中完整包含 must_not 中的某个表述时，才判定为违规

### contains_expected_points（要点覆盖）判定规则
- 逐条检查 expected_answer_contains 中的每个要点
- 使用宽松匹配：关键词的变形、包含该关键词的 longer phrase 均算命中
- 全部命中→true，有任何一个未命中→false

## 用例信息
【用例ID】{case['case_id']}
【测试类型】{case['test_type']}
【用户问题（最后一轮）】{case['user_query']}
【对话上下文】
{context_str if context_str else '无（单轮对话）'}

【期望意图】{case['expected_intent']}
【期望分支】{case['expected_branch']}
【期望答案应包含的关键要点】{expected_points}
【回复中禁止出现的表述】{must_not_str}
【期望投诉分级】{complaint_level}
【是否期望转人工】{case.get('expected_human_handoff', False)}
【边界值说明】{case.get('boundary_value') or '无'}

【实际客服回复】
{actual_response}

请从以下维度严格打分（0-5分），并输出 JSON 格式：
{{
    "intent_accuracy": <0-5>,
    "branch_routing": <0-5>,
    "answer_relevance": <0-5>,
    "information_completeness": <0-5>,
    "faithfulness": <0-5>,
    "tone_compliance": <0-5>,
    "must_not_violation": <true/false>,
    "boundary_correctness": <0-5>,
    "is_hallucination": <true/false>,
    "contains_expected_points": <true/false>,
    "handoff_correctness": <0-5, 不适用填-1>,
    "comment": "<80字以内>"
}}

只输出 JSON。"""

    try:
        resp = judge_client.chat.completions.create(
            model=JUDGE_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            response_format={"type": "json_object"}
        )
        score = json.loads(resp.choices[0].message.content)
    except Exception as e1:
        try:
            resp = judge_client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            content = resp.choices[0].message.content.strip()
            if content.startswith("```"):
                content = content.split("\n", 1)[1] if "\n" in content else content[3:]
            if content.endswith("```"):
                content = content.rsplit("```", 1)[0]
            content = content.strip()
            if content.startswith("json"):
                content = content[4:].strip()
            score = json.loads(content)
        except Exception as e2:
            score = {
                "error": str(e1), "intent_accuracy": 0, "branch_routing": 0,
                "answer_relevance": 0, "information_completeness": 0,
                "faithfulness": 0, "tone_compliance": 0,
                "must_not_violation": False, "boundary_correctness": 0,
                "is_hallucination": False, "contains_expected_points": False,
                "handoff_correctness": -1, "comment": f"裁判调用失败: {e1}"
            }

    # 确定性预检覆盖裁判误判
    must_not_list = case.get('must_not', [])
    expected_points_list = case.get('expected_answer_contains', [])

    code_must_not = check_must_not(actual_response, must_not_list)
    if not code_must_not and score.get('must_not_violation', False):
        score['must_not_violation'] = False
        orig_comment = score.get('comment', '')
        score['comment'] = f"[预检修正]must_not未违规(精确匹配)。{orig_comment}"

    code_points = check_expected_points(actual_response, expected_points_list)
    if code_points and not score.get('contains_expected_points', False):
        score['contains_expected_points'] = True
        orig_comment = score.get('comment', '')
        score['comment'] = f"[预检修正]要点已覆盖(宽松匹配)。{orig_comment}"

    return score

# ========== ★ 根因自动分类 ==========
def classify_root_cause(result: dict, case: dict) -> dict:
    """根据评测结果自动分类失败根因

    返回: {
        'rc_id': 'RC-1' ~ 'RC-5',
        'rc_name': 根因名称,
        'rc_detail': 具体描述,
        'rc_severity': 'P0'/'P1'/'P2'
    }
    """
    score = result.get('score', {})
    response = result.get('actual_response', '')
    branch_score = score.get('branch_routing', 5)
    intent_score = score.get('intent_accuracy', 5)
    faithfulness = score.get('faithfulness', 5)
    is_hallucination = score.get('is_hallucination', False)
    has_points = score.get('contains_expected_points', True)
    completeness = score.get('information_completeness', 5)
    is_multi_turn = result.get('is_multi_turn', False)
    must_not_violation = score.get('must_not_violation', False)
    expected_branch = case.get('expected_branch', '')

    # RC-5: 系统错误（最高优先级）
    if '[ERROR]' in response or 'BAD REQUEST' in response or '500' in response[:20]:
        return {
            'rc_id': 'RC-5', 'rc_name': '系统错误',
            'rc_detail': f'API返回错误: {response[:80]}',
            'rc_severity': 'P0'
        }

    # RC-1: 意图/分类器路由错误（branch_routing <= 2 或 intent_accuracy <= 2）
    if branch_score <= 2 or intent_score <= 2:
        # 细分路由错误子类
        if '退款' in case.get('user_query', '') and expected_branch == '商品咨询':
            rc_sub = '退款政策咨询被误路由到退款操作'
        elif '换' in case.get('user_query', '') and expected_branch == '退款换货':
            rc_sub = '换货意图被误识别为退货'
        elif '什么时候发货' in case.get('user_query', '') or '几天能到' in case.get('user_query', ''):
            rc_sub = '商品咨询被误分类到其他/订单查询'
        elif '投诉' in case.get('user_query', '') or '太慢' in case.get('user_query', ''):
            rc_sub = '投诉被误路由到其他分支'
        elif '改' in case.get('user_query', '') or '修改' in case.get('user_query', ''):
            rc_sub = '修改订单被误路由到其他分支'
        else:
            rc_sub = f'路由到错误分支 (branch={branch_score}, intent={intent_score})'
        return {
            'rc_id': 'RC-1', 'rc_name': '意图/分类器路由错误',
            'rc_detail': rc_sub,
            'rc_severity': 'P0'
        }

    # RC-2: 知识检索失败（触发兜底话术 + 未覆盖要点）
    fallback_patterns = [
        '知识库中暂未收录', '暂未找到相关信息', '未找到相关',
        '能否告诉我您感兴趣的具体商品名称', '建议您联系人工客服获取帮助'
    ]
    is_fallback = any(p in response for p in fallback_patterns)
    if is_fallback and not has_points:
        return {
            'rc_id': 'RC-2', 'rc_name': '知识检索失败',
            'rc_detail': f'触发兜底话术，知识库未命中 (completeness={completeness})',
            'rc_severity': 'P1'
        }

    # RC-4: 多轮上下文丢失（多轮场景 + 完整性低）
    if is_multi_turn and completeness <= 2:
        return {
            'rc_id': 'RC-4', 'rc_name': '多轮上下文丢失',
            'rc_detail': '多轮场景下商品/上下文信息未继承',
            'rc_severity': 'P1'
        }

    # RC-3a: LLM 幻觉
    if is_hallucination:
        return {
            'rc_id': 'RC-3a', 'rc_name': 'LLM幻觉',
            'rc_detail': f'回复包含虚构信息 (faithfulness={faithfulness})',
            'rc_severity': 'P1'
        }

    # RC-3b: LLM 生成缺陷 - 禁止表述违规
    if must_not_violation:
        return {
            'rc_id': 'RC-3b', 'rc_name': 'LLM禁止表述违规',
            'rc_detail': '回复包含禁止表述',
            'rc_severity': 'P1'
        }

    # RC-3c: LLM 生成缺陷 - 信息完整性不足
    if completeness <= 2 or (not has_points and branch_score >= 3):
        return {
            'rc_id': 'RC-3c', 'rc_name': 'LLM生成缺陷-信息不完整',
            'rc_detail': f'信息完整性={completeness}, 要点覆盖={has_points}',
            'rc_severity': 'P2'
        }

    # RC-3d: LLM 生成缺陷 - 其他
    if faithfulness <= 3 or branch_score <= 3:
        return {
            'rc_id': 'RC-3d', 'rc_name': 'LLM生成缺陷-其他',
            'rc_detail': f'忠实度={faithfulness}, 路由={branch_score}',
            'rc_severity': 'P2'
        }

    return {
        'rc_id': 'RC-3', 'rc_name': 'LLM生成缺陷',
        'rc_detail': '其他生成问题',
        'rc_severity': 'P2'
    }

# ========== 单条执行（含根因分类）==========
def evaluate_single_case(case: dict) -> dict:
    actual_response, conv_id = call_dify_multi_turn(case)
    score = llm_judge(case, actual_response)
    result = {
        "case_id": case["case_id"], "test_type": case["test_type"],
        "category": case["category"], "difficulty": case["difficulty"],
        "user_query": case["user_query"],
        "expected_intent": case["expected_intent"],
        "expected_branch": case["expected_branch"],
        "expected_if_else_path": ",".join(case.get("expected_if_else_path", [])),
        "expected_answer_contains": ",".join(case.get("expected_answer_contains", [])),
        "must_not": ",".join(case.get("must_not", [])),
        "expected_complaint_level": case.get("expected_complaint_level"),
        "expected_human_handoff": case.get("expected_human_handoff", False),
        "expected_retrieval_hit": case.get("expected_retrieval_hit"),
        "boundary_value": case.get("boundary_value"),
        "is_multi_turn": len(case.get("conversation_context", [])) > 0,
        "conversation_id": conv_id,
        "actual_response": actual_response,
        "score": score
    }

    # 判断是否通过（branch_routing >= 4 且 无禁止表述违规 且 无幻觉）
    is_pass = (
        score.get('branch_routing', 0) >= 4
        and not score.get('must_not_violation', False)
        and not score.get('is_hallucination', False)
    )
    result['is_pass'] = is_pass

    # 失败用例自动根因分类
    if not is_pass:
        rc = classify_root_cause(result, case)
        result['root_cause'] = rc
    else:
        result['root_cause'] = None

    return result

# ========== 断点续评 ==========
def load_checkpoint():
    path = os.path.join(OUTPUT_DIR, "checkpoint.json")
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return []

def save_checkpoint(results):
    path = os.path.join(OUTPUT_DIR, "checkpoint.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

print("✅ 函数定义完成（v5.0: 含根因自动分类 + V1/V2/V3 对比支持）")


In [ ]:
print("=== 连通性测试（第1条用例）===\n")
test_case = eval_cases[0]
result = evaluate_single_case(test_case)
print(f"用例: {test_case['user_query']}")
print(f"回复: {result['actual_response'][:200]}...")
print(f"\n评分: {json.dumps(result['score'], ensure_ascii=False, indent=2)}")
print(f"通过: {result['is_pass']}")
if result['root_cause']:
    print(f"根因: {result['root_cause']['rc_id']} - {result['root_cause']['rc_name']}")
print("\n✅ 连通性正常！继续运行下一个 Cell 批量评测" if '[ERROR]' not in result['actual_response'] else "❌ 连通性异常，请检查 Dify API 配置")


## Cell 6: 批量评测

支持断点续评，每10条自动保存。可修改 `START_INDEX` 和 `BATCH_SIZE` 控制评测范围。

In [ ]:
# ===== 可调参数 =====
START_INDEX = 0      # 从第几条开始（0-based）
BATCH_SIZE = 0       # 评测多少条（0=全部505条）
# ====================

results = load_checkpoint()
completed_ids = set(r["case_id"] for r in results)

# 确定评测范围
if BATCH_SIZE > 0:
    eval_range = eval_cases[START_INDEX:START_INDEX + BATCH_SIZE]
else:
    eval_range = eval_cases[START_INDEX:]

total_to_eval = len(eval_range)
already_done = len(completed_ids & set(c['case_id'] for c in eval_range))
print(f"本轮评测: {total_to_eval} 条（已完成 {already_done} 条，剩余 {total_to_eval - already_done} 条）")
print(f"历史断点: {len(results)} 条\n")

for i, case in enumerate(eval_range):
    if case["case_id"] in completed_ids:
        continue

    try:
        result = evaluate_single_case(case)
        results.append(result)
        completed_ids.add(case["case_id"])

        status = "✅" if result['is_pass'] else "❌"
        rc_info = ""
        if result['root_cause']:
            rc_info = f" [{result['root_cause']['rc_id']}:{result['root_cause']['rc_name']}]"

        print(f"  [{i+1}/{total_to_eval}] {status} {case['case_id']} | {case['category']} | "
              f"branch={result['score'].get('branch_routing', '?')}"
              f"{rc_info}")

    except Exception as e:
        print(f"  [{i+1}/{total_to_eval}] 💥 {case['case_id']} 异常: {e}")
        results.append({
            "case_id": case["case_id"], "test_type": case["test_type"],
            "category": case["category"], "difficulty": case["difficulty"],
            "user_query": case["user_query"],
            "expected_intent": case["expected_intent"],
            "expected_branch": case["expected_branch"],
            "is_multi_turn": len(case.get("conversation_context", [])) > 0,
            "actual_response": f"[EVAL_ERROR] {str(e)}",
            "score": {"error": str(e), "branch_routing": 0, "intent_accuracy": 0,
                      "is_hallucination": False, "must_not_violation": False,
                      "contains_expected_points": False, "information_completeness": 0,
                      "faithfulness": 0, "tone_compliance": 0, "boundary_correctness": 0,
                      "handoff_correctness": -1, "comment": f"评测异常: {e}"},
            "is_pass": False,
            "root_cause": {"rc_id": "RC-5", "rc_name": "系统错误",
                          "rc_detail": f"评测异常: {str(e)[:80]}", "rc_severity": "P0"}
        })
        completed_ids.add(case["case_id"])

    # 每10条保存一次
    if (i + 1) % 10 == 0:
        save_checkpoint(results)
        print(f"  --- 自动保存 ({len(results)} 条) ---")

    time.sleep(0.5)

# 最终保存
save_checkpoint(results)
print(f"\n✅ 评测完成！共 {len(results)} 条结果已保存到 {OUTPUT_DIR}/checkpoint.json")


In [ ]:
total = len(results)
rows = []
for r in results:
    s = r.get('score', {})
    rows.append({
        'case_id': r['case_id'],
        'test_type': r.get('test_type', ''),
        'category': r.get('category', ''),
        'difficulty': r.get('difficulty', ''),
        'user_query': r.get('user_query', ''),
        'expected_intent': r.get('expected_intent', ''),
        'expected_branch': r.get('expected_branch', ''),
        'actual_response': r.get('actual_response', ''),
        'is_pass': r.get('is_pass', False),
        'is_multi_turn': r.get('is_multi_turn', False),
        'intent_accuracy': s.get('intent_accuracy', 0),
        'branch_routing': s.get('branch_routing', 0),
        'answer_relevance': s.get('answer_relevance', 0),
        'information_completeness': s.get('information_completeness', 0),
        'faithfulness': s.get('faithfulness', 0),
        'tone_compliance': s.get('tone_compliance', 0),
        'boundary_correctness': s.get('boundary_correctness', 0),
        'is_hallucination': s.get('is_hallucination', False),
        'must_not_violation': s.get('must_not_violation', False),
        'contains_expected_points': s.get('contains_expected_points', False),
        'handoff_correctness': s.get('handoff_correctness', -1),
        'judge_comment': s.get('comment', ''),
        'rc_id': r.get('root_cause', {}).get('rc_id', '') if r.get('root_cause') else '',
        'rc_name': r.get('root_cause', {}).get('rc_name', '') if r.get('root_cause') else '',
        'rc_detail': r.get('root_cause', {}).get('rc_detail', '') if r.get('root_cause') else '',
        'rc_severity': r.get('root_cause', {}).get('rc_severity', '') if r.get('root_cause') else '',
    })

df = pd.DataFrame(rows)

# 标准化导出
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
csv_path = os.path.join(OUTPUT_DIR, f"eval_results_{timestamp}.csv")
json_path = os.path.join(OUTPUT_DIR, f"eval_results_{timestamp}.json")
std_csv = os.path.join(OUTPUT_DIR, f"eval_results_{BASELINE_ID}.csv")
std_json = os.path.join(OUTPUT_DIR, f"eval_results_{BASELINE_ID}.json")

df.to_csv(csv_path, index=False, encoding='utf-8-sig')
df.to_csv(std_csv, index=False, encoding='utf-8-sig')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
with open(std_json, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# 报告
passed = int(df['is_pass'].sum())
failed = total - passed
pass_rate = passed / total * 100 if total > 0 else 0

print("=" * 60)
print(f"📊 评测报告 v5.0 ({BASELINE_ID})")
print("=" * 60)
print(f"总用例数: {total}")
print(f"通过: {passed} | 失败: {failed} | 通过率: {pass_rate:.1f}%")
print(f"\n📁 文件已保存:")
print(f"   CSV: {csv_path}")
print(f"   JSON: {json_path}")
print(f"   标准化 CSV: {std_csv}")
print(f"   标准化 JSON: {std_json}")

# 按分支统计
print(f"\n--- 按分支 ---")
for branch in df['expected_branch'].unique():
    sub = df[df['expected_branch'] == branch]
    br_passed = int(sub['is_pass'].sum())
    br_total = len(sub)
    print(f"  {branch}: {br_passed}/{br_total} ({br_passed/br_total*100:.1f}%)")

# 按测试类型统计
print(f"\n--- 按测试类型 ---")
for ttype in df['test_type'].unique():
    sub = df[df['test_type'] == ttype]
    t_passed = int(sub['is_pass'].sum())
    t_total = len(sub)
    print(f"  {ttype}: {t_passed}/{t_total} ({t_passed/t_total*100:.1f}%)")

# 维度均分
print(f"\n--- 维度均分 ---")
for col in ['intent_accuracy', 'branch_routing', 'answer_relevance',
            'information_completeness', 'faithfulness', 'tone_compliance', 'boundary_correctness']:
    print(f"  {col}: {df[col].mean():.2f}")


## Cell 8: ★ 根因分类分析

对所有失败用例按根因自动分类，输出根因分布报告。

In [ ]:
failed_df = df[~df['is_pass']].copy()
passed_df = df[df['is_pass']].copy()

print("=" * 70)
print(f"📊 根因分类报告 ({BASELINE_ID})")
print("=" * 70)
print(f"\n失败用例: {len(failed_df)}/{len(df)} ({len(failed_df)/len(df)*100:.1f}%)")

# 根因分布
if len(failed_df) > 0:
    rc_counts = failed_df['rc_id'].value_counts()
    rc_names = failed_df.groupby('rc_id')['rc_name'].first()

    print(f"\n--- 根因分布 ---")
    for rc_id in rc_counts.index:
        count = rc_counts[rc_id]
        pct = count / len(failed_df) * 100
        rc_name = rc_names[rc_id]
        severity = failed_df[failed_df['rc_id'] == rc_id]['rc_severity'].iloc[0]
        print(f"  {rc_id} [{severity}] {rc_name}: {count} 条 ({pct:.1f}%)")

    # 根因 x 分支交叉分析
    print(f"\n--- 根因 x 分支交叉分析 ---")
    pivot = failed_df.pivot_table(
        index='rc_id', columns='expected_branch',
        values='case_id', aggfunc='count', fill_value=0
    )
    print(pivot.to_string())

    # 逐条失败用例详情
    print(f"\n--- 失败用例详情 ---")
    for _, row in failed_df.iterrows():
        print(f"\n  [{row['case_id']}] {row['category']} | {row['test_type']} | {row['difficulty']}")
        print(f"    Query: {row['user_query'][:50]}")
        print(f"    期望分支: {row['expected_branch']}")
        print(f"    根因: {row['rc_id']} - {row['rc_name']}")
        print(f"    详情: {row['rc_detail']}")
        print(f"    评分: branch={row['branch_routing']} intent={row['intent_accuracy']} "
              f"compl={row['information_completeness']} faith={row['faithfulness']}")
        print(f"    回复: {row['actual_response'][:100]}...")
        print(f"    裁判: {row['judge_comment'][:120]}")
else:
    print("\n🎉 全部通过，无失败用例！")


## Cell 9: ★ V1/V2/V3 三版本对比

对比三版评测结果，识别：
- **修复用例**: V1/V2 失败 → V3 通过
- **回归用例**: V1/V2 通过 → V3 失败
- **新增失败**: V3 新出现的失败
- **顽固失败**: V1+V2+V3 均失败

In [ ]:
# 当前 V3 结果
v3_fail_set = set(failed_df['case_id'].tolist())
v3_pass_set = set(passed_df['case_id'].tolist())

# 对比分析
v1_fixed = v1_fail_set & v3_pass_set       # V1失败 → V3通过
v1_persist = v1_fail_set & v3_fail_set      # V1失败 → V3仍失败
v1_regress = (set(c['case_id'] for c in eval_cases) - v1_fail_set) & v3_fail_set  # V1通过 → V3失败

v2_fixed = v2_fail_set & v3_pass_set        # V2失败 → V3通过
v2_persist = v2_fail_set & v3_fail_set       # V2失败 → V3仍失败
v2_regress = (set(c['case_id'] for c in eval_cases) - v2_fail_set) & v3_fail_set  # V2通过 → V3失败

# 顽固失败（V1+V2+V3 均失败）
stubborn = v1_fail_set & v2_fail_set & v3_fail_set

print("=" * 70)
print(f"📊 V1/V2/V3 三版本对比报告")
print("=" * 70)

print(f"\n--- 通过率对比 ---")
print(f"  V1 (baseline): {505-43}/505 = {(505-43)/505*100:.1f}%")
print(f"  V2 (P1优化):   {505-40}/505 = {(505-40)/505*100:.1f}%")
print(f"  V3 (P2修复):   {len(v3_pass_set)}/505 = {len(v3_pass_set)/505*100:.1f}%")

print(f"\n--- V1 → V3 对比 ---")
print(f"  ✅ 修复用例: {len(v1_fixed)} 条（V1失败→V3通过）")
print(f"  ❌ 顽固失败: {len(v1_persist)} 条（V1失败→V3仍失败）")
print(f"  ⚠️ 回归用例: {len(v1_regress)} 条（V1通过→V3失败）")

print(f"\n--- V2 → V3 对比 ---")
print(f"  ✅ 修复用例: {len(v2_fixed)} 条（V2失败→V3通过）")
print(f"  ❌ 顽固失败: {len(v2_persist)} 条（V2失败→V3仍失败）")
print(f"  ⚠️ 回归用例: {len(v2_regress)} 条（V2通过→V3失败）")

print(f"\n--- 顽固失败（V1+V2+V3 均失败）: {len(stubborn)} 条 ---")
for cid in sorted(stubborn):
    case = next((c for c in eval_cases if c['case_id'] == cid), None)
    row = failed_df[failed_df['case_id'] == cid]
    if case is not None and len(row) > 0:
        r = row.iloc[0]
        print(f"  {cid} | {case['category']} | {r['rc_id']}:{r['rc_name']} | {case['user_query'][:30]}")

if len(v1_fixed) > 0:
    print(f"\n--- V1→V3 修复用例详情: {len(v1_fixed)} 条 ---")
    for cid in sorted(v1_fixed):
        case = next((c for c in eval_cases if c['case_id'] == cid), None)
        if case:
            print(f"  ✅ {cid} | {case['category']} | {case['user_query'][:40]}")

if len(v1_regress) > 0 or len(v2_regress) > 0:
    all_regress = v1_regress | v2_regress
    print(f"\n--- ⚠️ 回归用例（V1或V2通过→V3失败）: {len(all_regress)} 条 ---")
    for cid in sorted(all_regress):
        case = next((c for c in eval_cases if c['case_id'] == cid), None)
        row = failed_df[failed_df['case_id'] == cid]
        if case is not None and len(row) > 0:
            r = row.iloc[0]
            print(f"  ⚠️ {cid} | {case['category']} | {r['rc_id']}:{r['rc_name']} | {case['user_query'][:30]}")
elif len(failed_df) == 0:
    print("\n🎉 V3 全部通过，无回归用例！")
else:
    print(f"\n✅ 无回归用例（V3 失败用例均为 V1/V2 已有失败）")


In [ ]:
total = len(results)
if total > 0:
    intent_correct = int((df['intent_accuracy'] >= 4).sum())
    branch_correct = int((df['branch_routing'] >= 4).sum())
    hallucination_count = int(df['is_hallucination'].sum())
    violation_count = int(df['must_not_violation'].sum())
    point_coverage = int(df['contains_expected_points'].sum()) / total

    meta_data = {
        "baseline_id": BASELINE_ID,
        "baseline_name": BASELINE_NAME,
        "created_at": datetime.now().isoformat(timespec='seconds'),
        "description": "P2硬性路由修复后全量评测，对比 baseline_v1/v2 验证优化效果",
        "p2_optimizations": [
            {"id": "P2-1", "desc": "退款政策咨询 vs 退款操作区分"},
            {"id": "P2-2", "desc": "什么时候发货误分类修复"},
            {"id": "P2-3", "desc": "换货意图识别与回复修正"},
            {"id": "P2-4", "desc": "投诉分支路由修复"},
            {"id": "P2-5", "desc": "退款操作 vs 修改订单区分"},
            {"id": "P2-6", "desc": "非电商场景超域拒答"},
            {"id": "P2-7", "desc": "边界混淆多意图路由优先级"},
        ],
        "eval_set": {
            "name": "eval_set_v4",
            "version": "4.0",
            "total_cases": total
        },
        "results": {
            "total_cases": total,
            "passed_cases": int(df['is_pass'].sum()),
            "failed_cases_count": int((~df['is_pass']).sum()),
            "pass_rate": round(df['is_pass'].sum() / total * 100, 1),
            "standardized_json": f"{OUTPUT_DIR}/eval_results_{BASELINE_ID}.json",
            "standardized_csv": f"{OUTPUT_DIR}/eval_results_{BASELINE_ID}.csv"
        },
        "metrics": {
            "overall": {
                "intent_accuracy": round(intent_correct / total, 3),
                "route_accuracy": round(branch_correct / total, 3),
                "point_coverage": round(point_coverage, 3),
                "hallucination_rate": round(hallucination_count / total, 3),
                "violation_rate": round(violation_count / total, 3),
                "avg_intent_score": round(df['intent_accuracy'].mean(), 2),
                "avg_route_score": round(df['branch_routing'].mean(), 2),
                "avg_relevance_score": round(df['answer_relevance'].mean(), 2),
                "avg_completeness_score": round(df['information_completeness'].mean(), 2),
                "avg_faithfulness_score": round(df['faithfulness'].mean(), 2),
                "avg_tone_score": round(df['tone_compliance'].mean(), 2),
                "avg_boundary_score": round(df['boundary_correctness'].mean(), 2)
            },
            "by_branch": {},
            "by_test_type": {},
            "root_cause_distribution": {}
        },
        "comparison": {
            "v1_pass_rate": round((505-43)/505*100, 1),
            "v2_pass_rate": round((505-40)/505*100, 1),
            "v3_pass_rate": round(df['is_pass'].sum() / total * 100, 1),
            "v1_fixed_count": len(v1_fixed),
            "v1_persist_count": len(v1_persist),
            "v1_regress_count": len(v1_regress),
            "v2_fixed_count": len(v2_fixed),
            "v2_persist_count": len(v2_persist),
            "v2_regress_count": len(v2_regress),
            "stubborn_failures": sorted(list(stubborn))
        },
        "judge_model": {
            "model": JUDGE_MODEL,
            "temperature": 0,
            "response_format": "json_object"
        }
    }

    # 按分支统计
    for branch in df['expected_branch'].unique():
        sub = df[df['expected_branch'] == branch]
        meta_data["metrics"]["by_branch"][branch] = {
            "pass_rate": round(sub['is_pass'].sum() / len(sub), 3),
            "total": len(sub),
            "passed": int(sub['is_pass'].sum()),
            "failed": int((~sub['is_pass']).sum())
        }

    # 按测试类型统计
    for ttype in df['test_type'].unique():
        sub = df[df['test_type'] == ttype]
        meta_data["metrics"]["by_test_type"][ttype] = {
            "pass_rate": round(sub['is_pass'].sum() / len(sub), 3),
            "total": len(sub),
            "failed": int((~sub['is_pass']).sum())
        }

    # 根因分布
    if len(failed_df) > 0:
        for rc_id in failed_df['rc_id'].unique():
            sub = failed_df[failed_df['rc_id'] == rc_id]
            meta_data["metrics"]["root_cause_distribution"][rc_id] = {
                "name": sub['rc_name'].iloc[0],
                "count": len(sub),
                "severity": sub['rc_severity'].iloc[0],
                "case_ids": sub['case_id'].tolist()
            }

    meta_path = f"{BASELINE_ID}_meta.json"
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(meta_data, f, ensure_ascii=False, indent=2)

    print(f"✅ 元数据已保存: {meta_path}")
    print(f"   通过率: {meta_data['results']['pass_rate']}%")
    print(f"   V1→V3 修复: {len(v1_fixed)} 条, 顽固: {len(v1_persist)} 条, 回归: {len(v1_regress)} 条")
    print(f"   V2→V3 修复: {len(v2_fixed)} 条, 顽固: {len(v2_persist)} 条, 回归: {len(v2_regress)} 条")
else:
    print("⚠️ 无评测结果，请先运行评测")


In [ ]:
ckpt_path = os.path.join(OUTPUT_DIR, "checkpoint.json")
if os.path.exists(ckpt_path):
    os.remove(ckpt_path)
    print(f"✅ 断点已清理: {ckpt_path}")
    print("现在可以重新运行 Cell 6 从头评测")
else:
    print("无断点文件，可直接评测")
